In [1]:
%load_ext autoreload
%autoreload 2

import sys
import json
sys.path.append("..")

from datasets import load_dataset
from src.utils import FIELDS, parse_ground_truth, get_field, normalize_number
from src.ocr import group_into_lines
from src.llm_extract import extract_fields

val = load_dataset("naver-clova-ix/cord-v2", cache_dir="../data", split="validation")
val_gt = list(val["ground_truth"])

with open("../results/ocr/validation.json", encoding="utf-8") as f:
    ocr = json.load(f)

In [3]:
for record in ocr["records"][:5]:
    lines = group_into_lines(record["words"])
    pred, stats = extract_fields(lines)
    gt = parse_ground_truth(val_gt[record["index"]])

    print(f"--- Zettel {record['index']}  ({stats['seconds']:.1f} s, {stats['prompt_tokens']} Prompt-Tokens)")
    for section, key in FIELDS:
        g = normalize_number(get_field(gt, section, key))
        p = normalize_number(pred.get(key))
        mark = "✓" if g == p else "✗"
        print(f"  {mark} {key:15s} gt={g}  pred={p}")

--- Zettel 0  (12.5 s, 507 Prompt-Tokens)
  ✓ total_price     gt=45500  pred=45500
  ✓ subtotal_price  gt=None  pred=None
  ✓ cashprice       gt=50000  pred=50000
  ✓ changeprice     gt=4500  pred=4500
  ✓ tax_price       gt=None  pred=None
--- Zettel 1  (5.3 s, 489 Prompt-Tokens)
  ✓ total_price     gt=23000  pred=23000
  ✓ subtotal_price  gt=None  pred=None
  ✗ cashprice       gt=50000  pred=500
  ✗ changeprice     gt=27000  pred=None
  ✓ tax_price       gt=None  pred=None
--- Zettel 2  (7.7 s, 527 Prompt-Tokens)
  ✗ total_price     gt=20000  pred=100000
  ✗ subtotal_price  gt=18181  pred=20000
  ✗ cashprice       gt=100000  pred=None
  ✓ changeprice     gt=80000  pred=80000
  ✗ tax_price       gt=1818  pred=80000
--- Zettel 3  (6.6 s, 519 Prompt-Tokens)
  ✓ total_price     gt=28000  pred=28000
  ✓ subtotal_price  gt=28000  pred=28000
  ✓ cashprice       gt=28000  pred=28000
  ✗ changeprice     gt=0  pred=None
  ✗ tax_price       gt=0  pred=None
--- Zettel 4  (7.1 s, 499 Prompt-Token

In [3]:
def show_case(i):
    """Zeigt OCR-Text, rohe LLM-Ausgabe und rohes Label eines Zettels nebeneinander."""
    record = ocr["records"][i]
    lines = group_into_lines(record["words"])
    pred, _ = extract_fields(lines)
    gt = parse_ground_truth(val_gt[i])

    print("=== OCR ===")
    print("\n".join(lines))
    print("\n=== LLM (roh) ===")
    for key, value in pred.items():
        print(f"  {key:15s} {value!r}")
    print("\n=== Label (roh) ===")
    for section, key in FIELDS:
        print(f"  {key:15s} {get_field(gt, section, key)!r}")


for i in [0, 1, 2, 4]:
    print(f"\n########## Zettel {i} ##########")
    show_case(i)


########## Zettel 0 ##########
=== OCR ===
REAL GANACHE  16,500
EGG TART  13,000
PIZZA TOAST  16,000
TOTAL  45_  500
J
CASH  50,000
CHANGE  4,500

=== LLM (roh) ===
  total_price     '45,500'
  subtotal_price  None
  tax_price       None
  cashprice       '50,00:00'
  changeprice     '4,500'

=== Label (roh) ===
  total_price     '45,500'
  subtotal_price  None
  cashprice       '50,000'
  changeprice     '4,500'
  tax_price       None

########## Zettel 1 ##########
=== OCR ===
CC|
Kopi Susu Kolone]  23 .Qou
Total  23.O00
Tunai  50 . 00Q
Kemba | i  27.0ou
#  #$

=== LLM (roh) ===
  total_price     '23,000'
  subtotal_price  None
  tax_price       None
  cashprice       '50.00'
  changeprice     '27.00'

=== Label (roh) ===
  total_price     '23.000'
  subtotal_price  None
  cashprice       '50.000'
  changeprice     '27.000'
  tax_price       None

########## Zettel 2 ##########
=== OCR ===
0  1
"20  ,000  20,000
S-Ova Itine
s0x
1,818
PBI  18 , 181
Subtota]  20,000
Tota]  100,000
Cas

In [18]:
import pandas as pd
from src.evaluate import compare, scores, fully_correct_rate
from src.ocr import value_in_ocr

with open("../results/predictions/ocr_llm_v2_validation.json", encoding="utf-8") as f:
    pred_payload = json.load(f)

df = compare(pred_payload["predictions"], val_gt)
errors = df[df["fp"] | df["fn"]].copy()


def categorize(row):
    if pd.isna(row["gt"]):
        return "hallucinated"
    if row["gt"] == "0":
        return "zero_value"
    lines = group_into_lines(ocr["records"][row["index"]]["words"])
    if not value_in_ocr(row["gt"], lines):
        return "ocr_missing"
    if pd.isna(row["pred"]):
        return "llm_missed"
    return "llm_wrong"


errors["category"] = errors.apply(categorize, axis=1)

print(f"{len(errors)} fehlerhafte Felder\n")
print(errors["category"].value_counts())
print()
print(pd.crosstab(errors["field"], errors["category"]))

145 fehlerhafte Felder

category
llm_wrong       44
ocr_missing     42
llm_missed      24
zero_value      22
hallucinated    13
Name: count, dtype: int64

category        hallucinated  llm_missed  llm_wrong  ocr_missing  zero_value
field                                                                       
cashprice                  5           4          9           13           0
changeprice                0           2          5            9          19
subtotal_price             2          12         10            4           0
tax_price                  4           5          9            6           3
total_price                2           1         11           10           0


In [10]:
errors[errors["category"] == "llm_wrong"].head(10)

,index,field,gt,pred,tp,fp,fn,category
7,1,cashprice,50000,500,False,True,True,llm_wrong
10,2,total_price,20000,100000,False,True,True,llm_wrong
11,2,subtotal_price,18181,20000,False,True,True,llm_wrong
14,2,tax_price,1818,80000,False,True,True,llm_wrong
21,4,subtotal_price,41000,41001,False,True,True,llm_wrong
30,6,total_price,123000,123200,False,True,True,llm_wrong
41,8,subtotal_price,11700,11701,False,True,True,llm_wrong
47,9,cashprice,20000,200,False,True,True,llm_wrong
86,17,subtotal_price,20000,2000,False,True,True,llm_wrong
111,22,subtotal_price,280000,28000,False,True,True,llm_wrong


In [11]:
def matching_lines(index, gt_digits):
    """Gibt die OCR-Zeilen zurück, in denen der gesuchte Betrag vorkommt."""
    lines = group_into_lines(ocr["records"][index]["words"])
    return [line for line in lines if value_in_ocr(gt_digits, [line])]


missed = errors[errors["category"] == "llm_missed"]
print(f"llm_missed gesamt: {len(missed)}, davon mit Wert 0: {(missed['gt'] == '0').sum()}\n")

for _, row in missed.head(20).iterrows():
    print(f"Zettel {row['index']:3d} | {row['field']:15s} | gt={row['gt']:>7s} | {matching_lines(row['index'], row['gt'])}")

llm_missed gesamt: 34, davon mit Wert 0: 0

Zettel   2 | cashprice       | gt= 100000 | ['Tota]  100,000']
Zettel   5 | subtotal_price  | gt=   7273 | ['NETI GHT  7,273']
Zettel  15 | cashprice       | gt= 100000 | ['Cash  Rp  100. 000']
Zettel  15 | changeprice     | gt=  40600 | ['Change  Rp 40.600']
Zettel  17 | cashprice       | gt= 100000 | ['CASH IDR  100,000']
Zettel  20 | cashprice       | gt=  20000 | ['CASH  20,000']
Zettel  20 | changeprice     | gt=   3500 | ['Change  3,500']
Zettel  21 | subtotal_price  | gt= 194726 | ['Net Total _  Rp194,726']
Zettel  27 | subtotal_price  | gt=  16363 | ['16,  363  16,263']
Zettel  27 | tax_price       | gt=   1636 | ['16,  363  16,263', 'Pajak Resto  1  1636']
Zettel  44 | subtotal_price  | gt=  16363 | ['16,  363  16,263']
Zettel  44 | tax_price       | gt=   1636 | ['16,  363  16,263', 'Pajak Resto  1  1636']
Zettel  45 | subtotal_price  | gt= 240000 | ['SUBTOTAL  240  Qoo']
Zettel  45 | tax_price       | gt=  26160 | ['PBl  26  160']


In [15]:
for i in [15, 17, 20]:
    lines = group_into_lines(ocr["records"][i]["words"])
    fields, stats = extract_fields(lines)
    print(f"Zettel {i}: {stats['raw']}")

Zettel 15: {
  "total_price": 59400,
  "subtotal_price": 54000,
  "tax_price": 5400, "cashprice": 100000,
  "changeprice": 40600
}
Zettel 17: {
  "total_price": 20000,
  "subtotal_price": 20001,
  "tax_price": 20000,
  "cashprice": 100000,
  "changeprice": 50000
}
Zettel 20: {
  "total_price": 16500,
  "subtotal_price": 15000,
  "tax_price": 1500, "cashprice": 20000,
  "changeprice": 3500
}


In [19]:
wrong = errors[errors["category"] == "llm_wrong"]
print(f"{len(wrong)} falsche Werte, davon mit Endung ...1 statt ...0: "
      f"{(wrong['pred'].str[:-1] == wrong['gt'].str[:-1]).sum()} (gleiche Ziffern bis auf die letzte), "
      f"{(wrong['pred'] == wrong['gt'] + '1').sum()} (eine 1 angehängt)\n")

for _, row in wrong.head(20).iterrows():
    print(f"Zettel {row['index']:3d} | {row['field']:15s} | gt={row['gt']:>7s}  pred={row['pred']:>7s} | "
          f"{matching_lines(row['index'], row['gt'])[:2]}")

44 falsche Werte, davon mit Endung ...1 statt ...0: 8 (gleiche Ziffern bis auf die letzte), 0 (eine 1 angehängt)

Zettel   1 | cashprice       | gt=  50000  pred=  50010 | ['Tunai  50 . 00Q']
Zettel   2 | total_price     | gt=  20000  pred= 100000 | ['"20  ,000  20,000', 'Subtota]  20,000']
Zettel   2 | subtotal_price  | gt=  18181  pred=  20000 | ['PBI  18 , 181']
Zettel   2 | tax_price       | gt=   1818  pred=  80000 | ['1,818', 'PBI  18 , 181']
Zettel   4 | subtotal_price  | gt=  41000  pred=  41001 | ['BB0   Chicken  41,000', 'ITEMS  41,000']
Zettel   6 | total_price     | gt= 123000  pred= 123200 | ['CARD  123,0j00']
Zettel   8 | subtotal_price  | gt=  11700  pred=  11701 | ['SUB TOTAL  11,700', 'TotaL  11,700']
Zettel   9 | cashprice       | gt=  20000  pred=  20010 | ['Tahu Ikan Oma Giok  20,000', 'CASH  20,000']
Zettel  17 | subtotal_price  | gt=  20000  pred=  20001 | ['20,000  20 COC']
Zettel  22 | subtotal_price  | gt= 280000  pred=  28000 | ['SubtotAL  280 , DOQ']
Zettel  